### Baselines

Se implementan tres políticas (Random, Most Popular e IPWLearner) y se estima su CTR con IPS (*Inverse Propensity Score*) usando los logs de la política `random` del Open Bandit Dataset.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from obp.dataset import OpenBanditDataset
from obp.policy import IPWLearner
from obp.ope import OffPolicyEvaluation, InverseProbabilityWeighting

OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
CAMPAIGN = "men"
SEED = 3633


Ahora se procede a separar el dataset de entrenamiento con validación (evaluación).

Los logs de `random` son separados según tiempo: el 70 % más antiguo para entrenar y el 30 % más reciente para evaluar.

In [5]:
dataset = OpenBanditDataset(behavior_policy="random", campaign=CAMPAIGN, data_path=OBD_PATH)
train, test = dataset.obtain_batch_bandit_feedback(test_size=0.3, is_timeseries_split=True)

n_actions, len_list = dataset.n_actions, dataset.len_list
pd.DataFrame({
    "Impresiones": [train["n_rounds"], test["n_rounds"]],
    "CTR": [train["reward"].mean(), test["reward"].mean()],
}, index=["Entrenamiento", "Evaluación"]).assign(Ítems=n_actions, Posiciones=len_list)

,Impresiones,CTR,Ítems,Posiciones
Entrenamiento,317064,0.004901,34,3
Evaluación,135885,0.005644,34,3


## Políticas

Cada política se representa como la probabilidad de mostrar cada ítem en cada posición (`action_dist`, de tamaño impresiones × ítems × posiciones).

- **Random:** elige cualquier ítem con la misma probabilidad.
- **Most Popular:** muestra en las posiciones los ítems con mayor CTR en entrenamiento.
- **IPWLearner:** política aprendida off-policy (`obp`): una regresión logística que, según las características del usuario, elige el ítem con mayor clic esperado.

In [6]:
n_test = test["n_rounds"]

# Random
dist_random = np.full((n_test, n_actions, len_list), 1 / n_actions)

# Most Popular
ctr_item = pd.Series(train["reward"]).groupby(train["action"]).mean().sort_values(ascending=False)
top_items = ctr_item.index[:len_list].to_numpy()
dist_popular = np.zeros((n_test, n_actions, len_list))
for pos, item in enumerate(top_items):
    dist_popular[:, item, pos] = 1

# IPWLearner
ipw_learner = IPWLearner(n_actions=n_actions, len_list=len_list,
                         base_classifier=LogisticRegression(max_iter=1000, random_state=SEED))
ipw_learner.fit(context=train["context"], action=train["action"], reward=train["reward"],
                pscore=train["pscore"], position=train["position"])
dist_ipw = ipw_learner.predict(context=test["context"])

print("Ítems con mayor CTR en entrenamiento:", top_items, ctr_item.head(len_list).round(4).to_list())

Ítems con mayor CTR en entrenamiento: [31 27 12] [0.0084, 0.0079, 0.0072]


## Evaluación con IPS

IPS estima el CTR de una política reponderando cada impresión del log por la probabilidad de que la política evaluada mostrara ese ítem, dividida por el propensity score. El intervalo de confianza de 95 % se obtiene por bootstrap.

In [7]:
ope = OffPolicyEvaluation(bandit_feedback=test, ope_estimators=[InverseProbabilityWeighting()])

politicas = {"Random": dist_random, "Most Popular": dist_popular, "IPWLearner": dist_ipw}
resultados = pd.DataFrame({
    nombre: ope.estimate_intervals(action_dist=dist, n_bootstrap_samples=1000, random_state=SEED)["ipw"]
    for nombre, dist in politicas.items()
}).T
resultados.columns = ["CTR estimado (IPS)", "IC 95% inferior", "IC 95% superior"]
resultados.round(5)

,CTR estimado (IPS),IC 95% inferior,IC 95% superior
Random,0.00564,0.00526,0.00604
Most Popular,0.00420,0.00225,0.00626
IPWLearner,0.00754,0.00500,0.01051


**Interpretación:**
- **Random:** el CTR estimado (0,00565) coincide con el CTR observado en evaluación (0,00564). Es lo esperado, porque los datos fueron generados por la misma política: cada peso de IPS vale 1 y la estimación se reduce al promedio de clics.
- **Most Popular:** obtiene un CTR estimado menor que Random (0,00425), pero su intervalo (0,00225–0,00651) contiene el valor de Random, por lo que la diferencia no es significativa.
- **IPWLearner:** obtiene el mayor CTR estimado (0,00751), pero su intervalo (0,00475–0,01026) también contiene el valor de Random.
- Los intervalos de Most Popular e IPWLearner son más amplios que el de Random. Ambas políticas eligen un único ítem por posición, así que IPS solo aprovecha las impresiones donde el ítem del log coincide con el elegido (cerca de 1 de cada 34), y cada una de ellas pesa 34 veces.